# 📝 Generador de posts de LinkedIn con Azure OpenAI (Structured Outputs + Pydantic)

**Reto proyecto — Desarrollo de Soluciones IA**

Chatbot por terminal que genera **posts de LinkedIn** a partir de una idea del usuario. Usa **Azure OpenAI** con **Structured Outputs** (`client.responses.parse()` + `text_format`) y un modelo **Pydantic** (`LinkedinPost`) para que la respuesta venga **estructurada y validada**. Incluye manejo robusto de errores: rechazos (*refusals*), validación, conexión y límites de tokens.

### Mapeo de la estructura de archivos pedida → celdas

```
├── main.py                  → Celda «Interfaz por terminal» (run_cli)
├── models/linkedin_post.py  → Celda «Modelo Pydantic»
├── core/api_client.py       → Celda «Cliente Azure OpenAI»
├── core/chatbot.py          → Celda «Lógica del chatbot»
├── requirements.txt         → Celda de dependencias
├── .env                     → Credenciales (privadas)
└── README.md                → Este encabezado
```

### Configuración (credenciales del lab)

Crea un archivo **`.env`** en la misma carpeta del notebook con la clave del lab que compartió el profesor:

```text
AZURE_OPENAI_API_KEY=pega_aqui_la_clave_del_lab
# Estos ya vienen por defecto en el código, solo cámbialos si usas tu propio recurso:
AZURE_OPENAI_ENDPOINT=https://marcosventosa0554426-resource.cognitiveservices.azure.com/
AZURE_OPENAI_DEPLOYMENT=gpt-4o
OPENAI_API_VERSION=2024-12-01-preview
```

- **Modelos/deployments del lab disponibles:** `gpt-4o`, `gpt-4o-mini`, `gpt-5.2-chat`, `gpt-5.4-mini`. Para **Structured Outputs** usa preferiblemente **`gpt-4o`**.
- **No subas el `.env`** al entregar; basta con el código.

> **Nota sobre la Responses API en Azure.** Este reto exige `responses.parse()`. La forma más fiable de usarla con el recurso del lab es el **cliente `OpenAI` apuntando al endpoint v1** (`.../openai/v1/`), que es la que el código usa por defecto. El cliente `AzureOpenAI` con `api_version=2024-12-01-preview` puede no exponer `/responses`. Si `responses.parse` te diera un 404, mira la nota al final de la sección 4 (alternativa con Chat Completions).


## 1. Dependencias (`requirements.txt`)

```text
openai
pydantic
python-dotenv
```


In [1]:
# Instalación de dependencias (ejecutar una vez)
%pip install -q openai pydantic python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Carga de la configuración de Azure (`.env`)

Cargamos las credenciales del recurso de Azure OpenAI. El endpoint, el deployment y la `api-version` traen por defecto los valores del lab; solo la **clave** debe ir en tu `.env` (es privada).


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # carga las variables del archivo .env

# Endpoint del recurso. Se NORMALIZA a la URL base (sin /openai ni /openai/v1),
# para construir luego la ruta correcta y evitar duplicar el path.
_raw_endpoint = os.getenv(
    "AZURE_OPENAI_ENDPOINT",
    "https://marcosventosa0554426-resource.cognitiveservices.azure.com/",
)
_base = _raw_endpoint.rstrip("/")
for _suf in ("/openai/v1", "/openai"):
    if _base.endswith(_suf):
        _base = _base[: -len(_suf)]
AZURE_ENDPOINT = _base + "/"           # p. ej. https://....cognitiveservices.azure.com/

API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
API_VERSION = os.getenv("OPENAI_API_VERSION", "2024-12-01-preview")

# En Azure, el "modelo" de las llamadas es el NOMBRE DEL DEPLOYMENT.
# El deployment 'gpt-4o' del lab corresponde a un modelo de la familia
# gpt-4o (versión 2024-08-06), que es uno de los modelos compatibles con
# Structured Outputs que cita el enunciado (gpt-4o-2024-08-06 / gpt-4.1 / gpt-5).
# Otros deployments del lab: gpt-4o-mini, gpt-5.2-chat, gpt-5.4-mini.
DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")

if API_KEY:
    print(f"✅ Configuración cargada. Endpoint base: {AZURE_ENDPOINT}")
    print(f"   Deployment: '{DEPLOYMENT}'  |  api-version: {API_VERSION}")
else:
    print("⚠️  Falta AZURE_OPENAI_API_KEY. Crea un .env con la clave del lab:")
    print("    AZURE_OPENAI_API_KEY=pega_aqui_la_clave_del_lab")


✅ Configuración cargada. Endpoint base: https://marcosventosa0554426-resource.cognitiveservices.azure.com/
   Deployment: 'gpt-4o'  |  api-version: 2024-12-01-preview


## 3. Modelo Pydantic (`models/linkedin_post.py`)

`LinkedinPost` define el esquema que la API debe respetar. Todos los campos son **obligatorios** y están tipados. Con `model_config = ConfigDict(extra="forbid")` prohibimos propiedades adicionales (equivale a `additionalProperties: false` en el JSON Schema), que es justo lo que exige el modo estricto de Structured Outputs. El SDK convierte esta clase a JSON Schema y **valida automáticamente** la respuesta.


In [3]:
from enum import Enum
from pydantic import BaseModel, ConfigDict, Field, field_validator


class CategoriaPost(str, Enum):
    """Categorías permitidas (enum compatible con Structured Outputs)."""
    tecnologia = "Tecnología"
    negocios = "Negocios"
    marketing = "Marketing"
    recursos_humanos = "Recursos Humanos"
    educacion = "Educación"
    finanzas = "Finanzas"
    emprendimiento = "Emprendimiento"
    desarrollo_personal = "Desarrollo personal"
    otros = "Otros"


class LinkedinPost(BaseModel):
    """Esquema estructurado de un post de LinkedIn."""

    # Configuración estricta: no se permiten campos fuera de los definidos
    model_config = ConfigDict(extra="forbid")

    title: str = Field(..., description="Título breve y atractivo del post (un gancho).")
    content: str = Field(..., description="Cuerpo del post listo para publicar, en español.")
    hashtags: list[str] = Field(..., description="Hashtags relevantes, formato #PalabraClave sin espacios.")
    category: CategoriaPost = Field(..., description="Categoría temática del post.")

    # --- Validaciones adicionales (se aplican a la respuesta, NO van al JSON Schema,
    #     así no rompen el modo estricto de Structured Outputs) ---

    @field_validator("title")
    @classmethod
    def _validar_title(cls, v: str) -> str:
        v = v.strip()
        if not (3 <= len(v) <= 150):
            raise ValueError("El título debe tener entre 3 y 150 caracteres.")
        return v

    @field_validator("content")
    @classmethod
    def _validar_content(cls, v: str) -> str:
        v = v.strip()
        if not (20 <= len(v) <= 3000):
            raise ValueError("El contenido debe tener entre 20 y 3000 caracteres.")
        return v

    @field_validator("hashtags")
    @classmethod
    def _normalizar_hashtags(cls, v: list[str]) -> list[str]:
        limpios: list[str] = []
        for h in v:
            h = h.strip().replace(" ", "")
            if h and not h.startswith("#"):
                h = "#" + h          # corrige los que vengan sin almohadilla
            if h:
                limpios.append(h)
        if not (1 <= len(limpios) <= 12):
            raise ValueError("Se esperaban entre 1 y 12 hashtags.")
        return limpios


## 4. Cliente Azure OpenAI con Structured Outputs (`core/api_client.py`)

`crear_cliente()` construye la URL correcta a partir del endpoint base (sin duplicar `/openai/v1`). Admite las dos opciones del lab:

- **`"openai_v1"` (por defecto):** cliente `OpenAI` sobre `.../openai/v1/`. Soporta la **Responses API**.
- **`"azure"`:** cliente `AzureOpenAI` clásico (endpoint + `api_version`).

`generar_post()` usa **`client.responses.parse()`** con `text_format=LinkedinPost`. Además, `generar_post_chat()` hace lo mismo con **Chat Completions** (`response_format=LinkedinPost`): se usa como **fallback automático** si la Responses API no está habilitada en el recurso (error 404), de modo que la solución funcione igualmente. En Azure, `model` es el **nombre del deployment**.


In [4]:
from openai import OpenAI, AzureOpenAI

SYSTEM_PROMPT = (
    "Eres un experto en redacción de contenido profesional para LinkedIn. "
    "A partir de la idea del usuario, redactas un post en ESPAÑOL, con un tono "
    "profesional y cercano. Devuelve: un 'title' atractivo (gancho), un 'content' "
    "listo para publicar (con saltos de línea y, si encaja, una llamada a la acción), "
    "una lista 'hashtags' de entre 5 y 8 etiquetas en formato #PalabraClave sin espacios, "
    "y una 'category' que resuma la temática. No incluyas nada fuera de esos campos."
)


def crear_cliente(modo: str = "openai_v1"):
    """Crea el cliente de Azure OpenAI.

    modo="openai_v1" (recomendado): cliente OpenAI sobre el endpoint v1 de Azure
                                     (soporta la Responses API).
    modo="azure":                   cliente AzureOpenAI clásico (usa api_version).
    """
    if modo == "azure":
        return AzureOpenAI(
            api_key=API_KEY,
            azure_endpoint=AZURE_ENDPOINT,     # URL base, sin /openai/v1
            api_version=API_VERSION,
        )
    return OpenAI(
        base_url=AZURE_ENDPOINT.rstrip("/") + "/openai/v1/",
        api_key=API_KEY,
    )


def generar_post(client, idea: str, deployment: str = DEPLOYMENT,
                 max_output_tokens: int = 800):
    """Vía principal: Responses API con Structured Outputs (responses.parse).

    Nota: en Azure, 'model' es el NOMBRE DEL DEPLOYMENT. Aquí 'gpt-4o' es un
    deployment de la familia gpt-4o-2024-08-06, modelo compatible con Structured
    Outputs (alineado con los modelos de ejemplo del enunciado).
    """
    return client.responses.parse(
        model=deployment,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": idea},
        ],
        text_format=LinkedinPost,
        max_output_tokens=max_output_tokens,
    )


def generar_post_chat(client, idea: str, deployment: str = DEPLOYMENT,
                      max_tokens: int = 800):
    """Fallback: Chat Completions con Structured Outputs (chat.completions.parse).

    Se usa si la Responses API no está habilitada en el recurso. El post validado
    queda en completion.choices[0].message.parsed.
    """
    return client.chat.completions.parse(
        model=deployment,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": idea},
        ],
        response_format=LinkedinPost,
        max_tokens=max_tokens,
    )


## 5. Lógica del chatbot (`core/chatbot.py`)

La clase `Chatbot` llama a la API, **gestiona los errores** y **muestra** el resultado. `generar()` traduce a mensajes claros: rechazos (*refusal*), respuesta incompleta (límite de tokens), `ValidationError` de Pydantic y errores del SDK (conexión, cuota, petición inválida). `mostrar_post()` presenta los cuatro campos de forma legible.


In [5]:
from pydantic import ValidationError
from openai import (
    APIError, APIConnectionError, RateLimitError, BadRequestError,
    NotFoundError, AuthenticationError, PermissionDeniedError,
)

try:
    from openai import LengthFinishReasonError
except ImportError:  # pragma: no cover
    class LengthFinishReasonError(Exception):
        pass


class Chatbot:
    """Genera y muestra posts de LinkedIn validados con Pydantic."""

    def __init__(self, client, deployment: str = DEPLOYMENT, max_output_tokens: int = 800):
        self.client = client
        self.deployment = deployment
        self.max_output_tokens = max_output_tokens

    @staticmethod
    def _buscar_refusal(response):
        """Refusal en una respuesta de la Responses API, o None."""
        for item in getattr(response, "output", []) or []:
            if getattr(item, "type", None) == "message":
                for parte in getattr(item, "content", []) or []:
                    if getattr(parte, "type", None) == "refusal":
                        return getattr(parte, "refusal", "(sin detalle)")
        return None

    def _procesar_responses(self, response):
        """Extrae el LinkedinPost de una respuesta de responses.parse()."""
        if getattr(response, "status", None) == "incomplete":
            motivo = getattr(getattr(response, "incomplete_details", None), "reason", "desconocido")
            print(f"⚠️  Respuesta incompleta (motivo: {motivo}). Sube 'max_output_tokens'.")
            return None
        refusal = self._buscar_refusal(response)
        if refusal:
            print(f"🚫 El modelo rechazó la petición: {refusal}")
            return None
        post = getattr(response, "output_parsed", None)
        if post is None:
            print("⚠️  La API no devolvió un post válido.")
        return post

    def _procesar_chat(self, completion):
        """Extrae el LinkedinPost de una respuesta de chat.completions.parse()."""
        mensaje = completion.choices[0].message
        if getattr(mensaje, "refusal", None):
            print(f"🚫 El modelo rechazó la petición: {mensaje.refusal}")
            return None
        post = getattr(mensaje, "parsed", None)
        if post is None:
            print("⚠️  La API no devolvió un post válido.")
        return post

    def generar(self, idea: str):
        """Devuelve un LinkedinPost validado, o None si hay error/refusal (ya informado)."""
        try:
            # Vía principal: Responses API
            try:
                response = generar_post(self.client, idea, self.deployment, self.max_output_tokens)
                return self._procesar_responses(response)
            except NotFoundError:
                # 404: o la Responses API no está habilitada, o el deployment no existe.
                print("ℹ️  La Responses API no está disponible; usando Chat Completions...")
                try:
                    completion = generar_post_chat(self.client, idea, self.deployment, self.max_output_tokens)
                    return self._procesar_chat(completion)
                except NotFoundError:
                    print(f"❌ El deployment '{self.deployment}' no existe en este recurso (404). "
                          "Revisa AZURE_OPENAI_DEPLOYMENT (p. ej. gpt-4o).")
                    return None

        except AuthenticationError:
            print("❌ Credenciales inválidas o caducadas (401). Revisa AZURE_OPENAI_API_KEY en el .env.")
            return None
        except PermissionDeniedError:
            print("❌ Sin permiso para este recurso o deployment (403). Revisa la clave y el deployment.")
            return None
        except LengthFinishReasonError:
            print("⚠️  La respuesta superó el límite de tokens. Aumenta 'max_output_tokens'.")
            return None
        except RateLimitError:
            print("⚠️  Límite de uso o cuota alcanzado (429). Inténtalo más tarde.")
            return None
        except APIConnectionError:
            print("⚠️  No se pudo conectar con Azure OpenAI. Revisa el endpoint y tu conexión.")
            return None
        except BadRequestError as e:
            # Distinguimos algunos 400 frecuentes por su código/mensaje
            mensaje = str(getattr(e, "message", "") or e)
            codigo = getattr(e, "code", None)
            if codigo == "content_filter" or "content_filter" in mensaje or "content management" in mensaje.lower():
                print("🚫 La petición se bloqueó por el filtro de contenido de Azure. Reformula la idea.")
            elif "DeploymentNotFound" in mensaje or "deployment" in mensaje.lower():
                print(f"❌ Deployment no válido: '{self.deployment}'. Revisa AZURE_OPENAI_DEPLOYMENT.")
            else:
                print(f"⚠️  Petición incorrecta (400): {e}")
            return None
        except ValidationError as e:
            print(f"⚠️  La respuesta no cumple el esquema esperado (validación Pydantic):\n{e}")
            return None
        except APIError as e:
            print(f"⚠️  Error de la API de Azure OpenAI: {e}")
            return None
        except Exception as e:
            print(f"⚠️  Error inesperado: {type(e).__name__}: {e}")
            return None

    @staticmethod
    def mostrar_post(post: "LinkedinPost", numero: int | None = None) -> None:
        """Muestra los cuatro campos de forma clara y organizada.

        numero: si se indica, numera el post (útil en sesiones con varias consultas).
        """
        categoria = getattr(post.category, "value", post.category)
        cabecera = "📋 POST GENERADO" + (f" #{numero}" if numero is not None else "")
        print("\n\n" + "=" * 62)
        print(cabecera)
        print("-" * 62)
        print(f"📌 TÍTULO\n   {post.title}")
        print(f"\n📝 CONTENIDO\n{post.content}")
        print(f"\n🏷️  HASHTAGS\n   {' '.join(post.hashtags)}")
        print(f"\n📂 CATEGORÍA: {categoria}")
        print("=" * 62 + "\n")


## 6. Interfaz por terminal (`main.py`)

Bucle que pide al usuario la idea del post, llama al chatbot y muestra el resultado. Permite **múltiples consultas** hasta que el usuario escribe `/salir`.


In [6]:
def run_cli(deployment: str = DEPLOYMENT, max_output_tokens: int = 800, modo: str = "openai_v1") -> None:
    """Interfaz por terminal para generar posts de LinkedIn con Azure OpenAI."""
    print("=" * 62)
    print("📝 Generador de posts de LinkedIn (Azure OpenAI + Structured Outputs)")
    print("=" * 62)

    if not API_KEY:
        print("❌ Falta AZURE_OPENAI_API_KEY. Crea un .env con la clave del lab y reejecuta.")
        return

    try:
        client = crear_cliente(modo=modo)
    except Exception as e:
        print(f"❌ No se pudo crear el cliente de Azure OpenAI: {e}")
        return

    bot = Chatbot(client, deployment=deployment, max_output_tokens=max_output_tokens)
    print(f"Deployment: {deployment}  |  cliente: {modo}")
    print("Describe la idea de tu post. Escribe '/salir' para terminar.")

    contador = 0
    while True:
        try:
            idea = input("\n💡 Tu idea para el post > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 ¡Hasta luego!")
            break

        if not idea:
            continue
        if idea.lower() in ("/salir", "/exit", "/quit", "salir"):
            print("👋 ¡Hasta luego!")
            break

        print("⏳ Generando post...")
        post = bot.generar(idea)
        if post is not None:
            contador += 1
            bot.mostrar_post(post, numero=contador)


def main() -> None:
    """Punto de entrada para usar desde terminal (main.py) con argumentos.

    Ejemplos:
        python main.py
        python main.py --deployment gpt-4o-mini --modo azure --max-tokens 1000
    """
    import argparse

    parser = argparse.ArgumentParser(
        description="Generador de posts de LinkedIn con Azure OpenAI (Structured Outputs)."
    )
    parser.add_argument("--deployment", default=DEPLOYMENT,
                        help="Nombre del deployment de Azure (p. ej. gpt-4o).")
    parser.add_argument("--modo", default="openai_v1", choices=["openai_v1", "azure"],
                        help="Tipo de cliente: 'openai_v1' (recomendado) o 'azure'.")
    parser.add_argument("--max-tokens", type=int, default=800, dest="max_output_tokens",
                        help="Máximo de tokens de salida.")
    # parse_known_args evita fallos si se llama dentro de un notebook
    args, _ = parser.parse_known_args()
    run_cli(deployment=args.deployment, max_output_tokens=args.max_output_tokens, modo=args.modo)


# En main.py, el punto de entrada sería:
#   if __name__ == "__main__":
#       main()


### ▶️ Opción A — Generar un post sin `input()` (recomendado en notebooks)

Crea el cliente y el bot, y genera un post directamente desde código. Útil si la caja de `input()` te resulta incómoda en el notebook (en VS Code aparece arriba del todo).

> Requiere `OPENAI_API_KEY` válida y saldo en la API.


In [7]:
# Generar un post de ejemplo directamente (sin interfaz interactiva)
cliente = crear_cliente()                 # modo "openai_v1" por defecto
bot = Chatbot(cliente, deployment=DEPLOYMENT)

idea = "Acabo de terminar un curso de desarrollo de soluciones de IA y quiero compartir lo aprendido"
print(f"💡 Idea: {idea}")
print("⏳ Generando post...")

post = bot.generar(idea)
if post is not None:
    bot.mostrar_post(post)
    # El objeto está validado; puedes acceder a cada campo: post.title, post.hashtags, ...


💡 Idea: Acabo de terminar un curso de desarrollo de soluciones de IA y quiero compartir lo aprendido
⏳ Generando post...


📋 POST GENERADO
--------------------------------------------------------------
📌 TÍTULO
   Descubre cómo la IA está transformando el futuro

📝 CONTENIDO
Hoy quiero compartirles el logro de haber completado un curso en el desarrollo de soluciones basadas en Inteligencia Artificial. Ha sido un viaje lleno de aprendizaje que me ha permitido entender cómo aplicar esta tecnología para resolver problemas innovadores y optimizar procesos. Estoy convencido de que la IA está redefiniendo la forma en que abordamos los desafíos actuales.

Me encantaría escuchar sus opiniones y experiencias sobre la implementación de IA en sus campos. ¡Juntos podemos explorar nuevas posibilidades!

🏷️  HASHTAGS
   #InteligenciaArtificial #DesarrolloTecnológico #Innovación #SolucionesIA #RevoluciónDigital #Aprendizaje #Superación

📂 CATEGORÍA: Tecnología



### ▶️ Opción B — Interfaz interactiva con `run_cli()`

Lanza el bucle por terminal. En una terminal real se ejecutaría con `python main.py`.


In [8]:
run_cli()


📝 Generador de posts de LinkedIn (Azure OpenAI + Structured Outputs)
Deployment: gpt-4o  |  cliente: openai_v1
Describe la idea de tu post. Escribe '/salir' para terminar.
⏳ Generando post...


📋 POST GENERADO #1
--------------------------------------------------------------
📌 TÍTULO
   Transforma el rechazo en impulso: claves para mejorar tu CV

📝 CONTENIDO
¿Has enviado tu CV y no has recibido respuesta? No estás solo en este desafío.

Primero, analiza si tu currículum refleja claramente tus logros y habilidades relevantes para el puesto. Evita descripciones genéricas y asegúrate de que los reclutadores puedan visualizar tu aporte al equipo.

Segundo, adapta tu CV para cada oferta laboral, resaltando las habilidades y experiencias específicas que buscan. Reúne cada detalle pertinente y demuéstrales que tu perfil encaja perfectamente.

Además, no subestimes la importancia de una carta de presentación. Es tu oportunidad de conectar con el empleador y explicar por qué eres la mejor opció

## 7. (Opcional) Prueba del manejo de errores sin llamar a la API

Esta celda **no forma parte de la solución entregable**: comprueba, sin gastar API, que (a) la **validación Pydantic** detecta datos incorrectos, (b) la detección de **refusal** funciona, y (c) `mostrar_post()` formatea bien un post válido.


In [9]:
from types import SimpleNamespace

# (a) Validación Pydantic: faltan campos obligatorios -> ValidationError
try:
    LinkedinPost(title="Solo título")  # faltan content, hashtags y category
except ValidationError as e:
    print("✅ Validación Pydantic detectó el error (faltan campos):")
    print("  ", e.error_count(), "errores de validación\n")

# (b) Detección de refusal con una respuesta simulada
respuesta_simulada = SimpleNamespace(
    output=[SimpleNamespace(
        type="message",
        content=[SimpleNamespace(type="refusal", refusal="No puedo ayudar con eso.")],
    )]
)
refusal = Chatbot._buscar_refusal(respuesta_simulada)
print("✅ Refusal detectado:", refusal, "\n")

# (c) Formateo de un post válido
ejemplo = LinkedinPost(
    title="3 lecciones tras mi primer proyecto de IA",
    content="Esta semana terminé mi primer proyecto de IA y me llevo varias lecciones...\n¿Tú qué añadirías?",
    hashtags=["#IA", "#Aprendizaje", "#Tecnologia"],
    category="Tecnología",
)
Chatbot.mostrar_post(ejemplo)


✅ Validación Pydantic detectó el error (faltan campos):
   3 errores de validación

✅ Refusal detectado: No puedo ayudar con eso. 



📋 POST GENERADO
--------------------------------------------------------------
📌 TÍTULO
   3 lecciones tras mi primer proyecto de IA

📝 CONTENIDO
Esta semana terminé mi primer proyecto de IA y me llevo varias lecciones...
¿Tú qué añadirías?

🏷️  HASHTAGS
   #IA #Aprendizaje #Tecnologia

📂 CATEGORÍA: Tecnología

